<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/03_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3 - Build the training set, interrogate the features

Three questions, each with a figure that can embarrass us.

1. **Is the quality filter neutral?** Negative median reflectance is a Collection 1
   aerosol over-correction. If it happens preferentially over dark water, dropping those
   rows drops the clear end of the Secchi distribution, and every model in this lineage
   trained on a biased sample. F10 reports the mean Secchi on both sides of every filter
   step.
2. **Are the 15 band ratios independent?** They are algebraic functions of 6 medians.
   F11 gives the condition number. This is why Phase 4 reports permutation importance
   rather than Gini.
3. **Is a forty-year clarity trend real, or is it the satellites?** F13 plots band
   reflectance for a large, clear, stable lake. It should be flat. If it steps at 2013,
   any uncorrected trend is an instrument artifact.

In [ ]:
# Cell 1 of 5: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

# Force a fresh import of the freshly pulled code. Colab keeps modules from a
# previous run in memory, so a git pull updates the files on disk while the
# kernel keeps running the old lakeclarity and silently ignores any fix. Purge
# them here so the import in the next cell always loads the pulled version.
for _stale in [m for m in list(sys.modules)
               if m == 'lakeclarity' or m.startswith('lakeclarity.')]:
    del sys.modules[_stale]


In [ ]:
# Cell 2 of 5: the filter chain, with the target lakes removed first
import pandas as pd
from lakeclarity import config, explore, features, locus, viz

viz.use_style()

region = pd.read_parquet(config.INTERIM_DIR / "region_matchups.parquet")
targets = pd.read_csv(config.TABLE_DIR / "T05_target_lakes.csv", index_col=0).squeeze()
holdout = [int(targets["large"]), int(targets["small"])]
print("holding out lakes:", holdout)

train, filter_log = features.build_training_frame(region, holdout_lake_ids=holdout)
waterfall = filter_log.to_frame()
waterfall.to_csv(config.TABLE_DIR / "T06_filter_waterfall.csv", index=False)
print(waterfall.to_string(index=False))

train = locus.attach_lake_metadata(train)
train.to_parquet(config.INTERIM_DIR / "train.parquet", index=False)
print(f"\ntraining set: {len(train):,} matchups on {train.lagoslakeid.nunique():,} lakes")
assert not set(holdout) & set(train["lagoslakeid"]), "target lake leaked into training"

In [ ]:
# Cell 3 of 5: does the QA filter delete clear water?
shift = waterfall.loc[waterfall["step"] == "negative_reflectance", "secchi_shift_m"]
if len(shift) and shift.iloc[0] < -0.02:
    print(f"CONFIRMED: dropping negative-reflectance rows lowers mean Secchi by "
          f"{abs(shift.iloc[0]):.3f} m.")
    print("The standard quality filter is biased against clear water, which is the water")
    print("this model is asked to predict. Report this.")
elif len(shift) and shift.iloc[0] > 0.02:
    print(f"REFUTED: the filter *raises* mean Secchi by {shift.iloc[0]:.3f} m.")
    print("Negative reflectance is concentrated in turbid water, not clear. The Phase 1")
    print("hypothesis was wrong; say so in the report rather than quietly dropping it.")
else:
    print("NEUTRAL: the filter does not measurably shift the Secchi distribution.")

In [ ]:
# Cell 4 of 5: collinearity and univariate signal
X = features.feature_matrix(train)
y = train[config.LOG_TARGET]

cond = explore.predictor_condition_number(X)
print(f"condition number of the standardised predictor matrix: {cond:,.0f}")
print("Gini importance is not interpretable on a matrix this collinear. Phase 4 uses")
print("permutation importance and shows both side by side.")

mi = explore.mutual_information(X, y)
mi.to_csv(config.TABLE_DIR / "T07_mutual_information.csv")
print("\ntop 8 predictors by mutual information with log10 Secchi:")
print(mi.head(8).to_string())

top_is_blue = any(c in config.CDOM_RATIOS for c in mi.head(3).index)
print(f"\nblue-family ratio in the top three: {top_is_blue}")
print("CDOM absorbs in the blue and barely scatters, so blue-family ratios should lead in")
print("stained glacial lakes. If red and NIR lead instead, this region is sediment-driven")
print("and the analogy to New Hampshire is weaker than assumed. That is worth knowing now.")

In [ ]:
# Cell 5 of 5: figures F10 to F14
ref_lake = explore.pick_stable_reference_lake(train)
print("drift reference lake:", train.loc[train.lagoslakeid == ref_lake, "lake_name"].iloc[0])

for fig, fid, slug in [
    (explore.fig_filter_waterfall(filter_log), "F10", "filter_waterfall"),
    (explore.fig_correlation_heatmap(X), "F11", "predictor_collinearity"),
    (explore.fig_mutual_information(X, y), "F12", "mutual_information"),
    (explore.fig_stable_lake_drift(train, ref_lake), "F13", "stable_lake_drift"),
    (explore.fig_seasonality(train), "F14", "seasonality"),
]:
    print("wrote", viz.save(fig, fid, slug).name)